Before executing the code, use start-cluster script. After, to deallocate the memory, use stop-cluster script

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import count, max, avg, min, desc
from dotenv import load_dotenv
import os

load_dotenv()
POSTGRESQL_JDBC_PATH=os.getenv('POSTGRESQL_JDBC_PATH')
USER_POSTGRESQL=os.getenv('USER_POSTGRESQL')
PASSWORD_POSTGRESQL=os.getenv('PASSWORD_POSTGRESQL')
URL_POSTGRES=os.getenv('URL_POSTGRES')

In [2]:
spark = SparkSession.builder\
        .appName('Spark_DB_Quering')\
        .master('spark://127.0.0.1:7077')\
        .config("spark.driver.host", "127.0.0.1")\
        .config("spark.driver.bindAddress", "127.0.0.1")\
        .config('spark.jars', POSTGRESQL_JDBC_PATH)\
        .getOrCreate()

26/08/27 00:14:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/mihailmoroz/.pyenv/versions/env1/lib/python3.11/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


I decided to change number of partitions on shuffle,because the amount of data is quite small for default 200 partitions.
I guess that creating many small partitions is an overkill.
And inspecting the plan, with default 200 set, AQE was combining partitions, so why not change the default?

In [3]:

spark.conf.set("spark.sql.shuffle.partitions", "40")

In [4]:
def load_table(table_name):
    df = spark.read\
              .format("jdbc")\
              .option("url", URL_POSTGRES)\
              .option("dbtable", table_name)\
              .option("user", USER_POSTGRESQL)\
              .option("password", PASSWORD_POSTGRESQL)\
              .option("driver", "org.postgresql.Driver")\
              .load()
    return df

def load_and_cache_table(table_name):
    df = load_table(table_name)
    cached_df = df.cache()

    return cached_df

table_names = ["category","film_category","actor","film_actor","inventory","rental","payment","address","city"]
dataframes = {}
for table in table_names:
    #those tables are used twice or more, so i decided to cache them, especially as they are all relatively small
    dataframes[table] = load_and_cache_table(table_name=table)


1st query

Output the number of movies in each category, sorted descending.

In [5]:
df_movie_num_per_category = dataframes["category"].join(dataframes["film_category"], dataframes["category"].category_id == dataframes["film_category"].category_id, "inner")\
                            .groupBy(dataframes["category"].category_id, dataframes["category"].name).agg(count(dataframes["category"].name).alias('cnt'))\
                            .sort(desc('cnt')).select('name', 'cnt')

df_movie_num_per_category.show()

+-----------+---+
|       name|cnt|
+-----------+---+
|     Sports| 74|
|    Foreign| 73|
|     Family| 69|
|Documentary| 68|
|  Animation| 66|
|     Action| 64|
|        New| 63|
|      Drama| 62|
|      Games| 61|
|     Sci-Fi| 61|
|   Children| 60|
|     Comedy| 58|
|     Travel| 57|
|   Classics| 57|
|     Horror| 56|
|      Music| 51|
+-----------+---+



2nd query

Output the 10 actors whose movies rented the most, sorted in descending order.

In [6]:
from pyspark.sql.functions import concat, lit

df_top10_actors = dataframes["actor"].join(dataframes["film_actor"], dataframes["actor"].actor_id == dataframes["film_actor"].actor_id, "left")
df_top10_actors = df_top10_actors.join(dataframes["inventory"], dataframes["inventory"].film_id == df_top10_actors.film_id, "left")
df_top10_actors = df_top10_actors.join(dataframes["rental"], dataframes["rental"].inventory_id == df_top10_actors.inventory_id, "left")\
                  .groupBy(dataframes["actor"].actor_id, 'first_name', 'last_name')\
                  .agg(count('rental_id').alias('movie_cnt'))\
                  .sort(desc('movie_cnt'))\
                  .select('actor_id', concat('first_name', lit(' '), 'last_name').alias('name'), 'movie_cnt')

df_top10_actors.limit(10).show()

+--------+------------------+---------+
|actor_id|              name|movie_cnt|
+--------+------------------+---------+
|     107|    Gina Degeneres|      753|
|     181|    Matthew Carrey|      678|
|     198|       Mary Keitel|      674|
|     144|Angela Witherspoon|      654|
|     102|       Walter Torn|      640|
|      60|       Henry Berry|      612|
|     150|       Jayne Nolte|      611|
|      37|        Val Bolger|      605|
|      23|     Sandra Kilmer|      604|
|      90|      Sean Guiness|      599|
+--------+------------------+---------+



3rd query

Output the category of movies on which the most money was spent.

In [7]:
from pyspark.sql.functions import coalesce, sum
df_most_spent_category = dataframes["category"].join(dataframes["film_category"], dataframes["category"].category_id == dataframes["film_category"].category_id, "left")
df_most_spent_category = df_most_spent_category.join(dataframes["inventory"], dataframes["inventory"].film_id == df_most_spent_category.film_id, "left")
df_most_spent_category = df_most_spent_category.join(dataframes["rental"], dataframes["rental"].inventory_id == df_most_spent_category.inventory_id, "left")
df_most_spent_category = df_most_spent_category.join(dataframes["payment"], dataframes["payment"].rental_id == df_most_spent_category.rental_id, "left")

df_most_spent_category = df_most_spent_category.groupBy(dataframes["category"].category_id, dataframes["category"].name)\
                         .agg(coalesce(sum(df_most_spent_category.amount), lit(0)).alias('amt_spent'))\
                         .sort(desc('amt_spent'))

df_most_spent_category.limit(1).show()

+-----------+------+---------+
|category_id|  name|amt_spent|
+-----------+------+---------+
|         15|Sports|  4892.19|
+-----------+------+---------+



In [8]:
dataframes["film"] = load_table("film")

4th query

Print the names of movies that are not in the inventory. Write a query without using the IN operator.

In [9]:
df_movies_not_in_inventory = dataframes["film"].join(dataframes["inventory"], dataframes["inventory"].film_id==dataframes["film"].film_id, "anti")\
                             .select('film_id', 'title')

df_movies_not_in_inventory.show()

+-------+--------------------+
|film_id|               title|
+-------+--------------------+
|     14|      Alice Fantasia|
|     33|         Apollo Teen|
|     36|      Argonauts Town|
|     38|       Ark Ridgemont|
|     41|Arsenic Independence|
|     87|   Boondock Ballroom|
|    108|       Butch Panther|
|    128|       Catch Amistad|
|    144| Chinatown Gladiator|
|    148|      Chocolate Duck|
|    171|Commandments Express|
|    192|    Crossing Divorce|
|    195|     Crowds Telemark|
|    198|    Crystal Breaking|
|    217|          Dazed Punk|
|    221|Deliverance Mulho...|
|    318|   Firehouse Vietnam|
|    325|       Floats Garden|
|    332|Frankenstein Stra...|
|    359|  Gladiator Westward|
+-------+--------------------+
only showing top 20 rows


Creating an intermediate dataframe that will be used for query 5

In [10]:
df_actor_film_count = dataframes["actor"].join(dataframes["film_actor"], on="actor_id", how="inner")
df_actor_film_count = df_actor_film_count.join(dataframes["film_category"], on="film_id", how="inner")
df_actor_film_count = df_actor_film_count.join(dataframes["category"], on="category_id", how="inner")

df_actor_film_count = df_actor_film_count.filter(df_actor_film_count.name=='Children')\
                      .groupBy('actor_id', 'first_name', 'last_name').agg(count('film_id').alias('cnt'))\
                      .select('actor_id', 'first_name', 'last_name', 'cnt')

df_actor_film_count.show()

+--------+-----------+---------+---+
|actor_id| first_name|last_name|cnt|
+--------+-----------+---------+---+
|      59|     Dustin|   Tautou|  1|
|       1|   Penelope|  Guiness|  1|
|      44|       Nick| Stallone|  2|
|      72|       Sean| Williams|  1|
|     110|      Susan|    Davis|  2|
|     163|Christopher|     West|  3|
|     198|       Mary|   Keitel|  2|
|      37|        Val|   Bolger|  4|
|      52|     Carmen|     Hunt|  3|
|      79|        Mae|  Hoffman|  1|
|      80|      Ralph|     Cruz|  5|
|     120|   Penelope|   Monroe|  3|
|     164|   Humphrey|   Willis|  1|
|      22|      Elvis|     Marx|  2|
|      29|       Alec|    Wayne|  3|
|      81|   Scarlett|    Damon|  4|
|     115|   Harrison|     Bale|  1|
|     139|       Ewan|  Gooding|  1|
|      96|       Gene|   Willis|  1|
|      98|      Chris|  Bridges|  2|
+--------+-----------+---------+---+
only showing top 20 rows


Another intermediate helper dataframe

In [11]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank
df_ranked_actor_film_count = df_actor_film_count.withColumn("ranking", dense_rank().over(Window.orderBy(desc('cnt')))).select("ranking","actor_id", "cnt", concat("first_name", lit(' '), "last_name").alias('name'))

df_ranked_actor_film_count.show()

26/08/27 00:16:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/27 00:16:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/27 00:16:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/27 00:16:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/27 00:16:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/27 00:16:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/27 0

+-------+--------+---+----------------+
|ranking|actor_id|cnt|            name|
+-------+--------+---+----------------+
|      1|      17|  7|    Helen Voight|
|      2|      80|  5|      Ralph Cruz|
|      2|     127|  5|   Kevin Garland|
|      2|      66|  5|      Mary Tandy|
|      2|     140|  5|     Whoopi Hurt|
|      3|      37|  4|      Val Bolger|
|      3|      81|  4|  Scarlett Damon|
|      3|      58|  4|Christian Akroyd|
|      3|      92|  4|  Kirsten Akroyd|
|      3|     150|  4|     Jayne Nolte|
|      3|     131|  4|    Jane Jackman|
|      3|     142|  4|      Jada Ryder|
|      3|     101|  4|     Susan Davis|
|      3|      13|  4|        Uma Wood|
|      3|      93|  4|   Ellen Presley|
|      3|     173|  4|   Alan Dreyfuss|
|      3|      23|  4|   Sandra Kilmer|
|      3|     109|  4|  Sylvester Dern|
|      3|     187|  4|      Renee Ball|
|      4|     102|  3|     Walter Torn|
+-------+--------+---+----------------+
only showing top 20 rows


26/08/27 00:16:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/27 00:16:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/27 00:16:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


5th query

Output the top 3 actors who have appeared the most in movies in the “Children” category. 

If several actors have the same number of movies, output all of them.

In [12]:
df_top3_actors_children_category = df_ranked_actor_film_count.filter(df_ranked_actor_film_count.ranking<=3)

df_top3_actors_children_category.show()

26/08/27 00:16:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/27 00:16:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/27 00:16:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/27 00:16:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/27 00:16:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/27 00:16:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/27 0

+-------+--------+---+----------------+
|ranking|actor_id|cnt|            name|
+-------+--------+---+----------------+
|      1|      17|  7|    Helen Voight|
|      2|      80|  5|      Ralph Cruz|
|      2|     127|  5|   Kevin Garland|
|      2|      66|  5|      Mary Tandy|
|      2|     140|  5|     Whoopi Hurt|
|      3|      37|  4|      Val Bolger|
|      3|      81|  4|  Scarlett Damon|
|      3|      92|  4|  Kirsten Akroyd|
|      3|     150|  4|     Jayne Nolte|
|      3|     131|  4|    Jane Jackman|
|      3|      58|  4|Christian Akroyd|
|      3|     101|  4|     Susan Davis|
|      3|      13|  4|        Uma Wood|
|      3|      93|  4|   Ellen Presley|
|      3|     173|  4|   Alan Dreyfuss|
|      3|     142|  4|      Jada Ryder|
|      3|      23|  4|   Sandra Kilmer|
|      3|     109|  4|  Sylvester Dern|
|      3|     187|  4|      Renee Ball|
+-------+--------+---+----------------+



26/08/27 00:16:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/27 00:16:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/27 00:16:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/27 00:16:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


6th query

Output cities with the number of active and inactive customers (active - customer.active = 1). 

Sort by the number of inactive customers in descending order.

In [13]:
#forgot to read customer table
dataframes["customer"] = load_table("customer")

In [14]:
from pyspark.sql.functions import when
df_active_statistics = dataframes["city"].join(dataframes["address"], on="city_id", how="left")
df_active_statistics = df_active_statistics.join(dataframes["customer"], on="address_id", how="left")

df_active_statistics = df_active_statistics.withColumn("inactive_flag", when(df_active_statistics.active == 0, 1).otherwise(0))
df_active_statistics = df_active_statistics.groupBy('city_id', 'city', 'active')\
                       .agg(sum('active').alias('active_cnt'),
                            sum('inactive_flag').alias('inactive_cnt'))\
                       .sort(desc('inactive_cnt'))
df_active_statistics = df_active_statistics.select('city_id', 'city', 'active_cnt', 'inactive_cnt')

df_active_statistics.show()


+-------+------------------+----------+------------+
|city_id|              city|active_cnt|inactive_cnt|
+-------+------------------+----------+------------+
|    259|            Kamyin|         0|           1|
|    512|       Szkesfehrvr|         0|           1|
|    356|         Najafabad|         0|           1|
|     24|            Amroha|         0|           1|
|    139|            Daxian|         0|           1|
|    283|        Kumbakonam|         0|           1|
|    125|     Coatzacoalcos|         0|           1|
|     57|           Bat Yam|         0|           1|
|    281|            Ktahya|         0|           1|
|    577|           Wroclaw|         0|           1|
|    495|   Southend-on-Sea|         0|           1|
|    578|          Xiangfan|         0|           1|
|    111|  Charlotte Amalie|         0|           1|
|    554|          Uluberia|         0|           1|
|    407|         Pingxiang|         0|           1|
|    148|          Duisburg|         1|       

7th query 

Output the category of movies that have the highest number of total rental hours in the city (customer.address_id in this city) and that start with the letter “a”. 

Do the same for cities that have a “-” in them. Write everything in one query.

In [ ]:
from pyspark.sql.functions import datediff
df_category_rank_per_city = dataframes["category"].join(dataframes["film_category"], on="category_id", how="inner")
df_category_rank_per_city = df_category_rank_per_city.join(dataframes["inventory"], on="film_id", how="inner")\
                            .join(dataframes["rental"], on="inventory_id", how="inner")\
                            .join(dataframes["customer"], on="customer_id", how="inner")\
                            .join(dataframes["address"], on="address_id", how="inner")\
                            .join(dataframes["city"], on="city_id", how="inner")

df_category_rank_per_city = df_category_rank_per_city.where(df_category_rank_per_city.city.ilike('A%') | df_category_rank_per_city.city.like('%-%'))\
                            .groupBy("city_id", "city", "category_id", "name")\
                            .agg(coalesce(sum(datediff('return_date', 'rental_date')), lit(0)).alias("rent_time_per_category"))\
                            .withColumn("ranking", dense_rank().over(Window.partitionBy('city_id').orderBy(desc("rent_time_per_category"))))\
                            .select("category_id", "name", "city_id", "city", "rent_time_per_category", "ranking")

df_category_rank_per_city.where(df_category_rank_per_city['ranking']==1).show()

+-----------+-----------+-------+--------------------+----------------------+-------+
|category_id|       name|city_id|                city|rent_time_per_category|ranking|
+-----------+-----------+-------+--------------------+----------------------+-------+
|          6|Documentary|      1|  A Corua (La Corua)|                    24|      1|
|          5|     Comedy|      1|  A Corua (La Corua)|                    24|      1|
|         14|     Sci-Fi|      2|                Abha|                    23|      1|
|         14|     Sci-Fi|      3|           Abu Dhabi|                    21|      1|
|          7|      Drama|      4|                Acua|                    23|      1|
|          5|     Comedy|      5|               Adana|                    22|      1|
|          8|     Family|      6|         Addis Abeba|                    21|      1|
|         13|        New|      7|                Aden|                    33|      1|
|          3|   Children|      8|               Adoni|

In [ ]:
spark.stop()